### Practice 1

# Exercise 1: Pipeline Persistence

## Objective
In this exercise, we train a Scikit-learn Pipeline using 3 input features, save it, load it again, and test its behavior when the input contains 8 features.

### Main Question
Can a Pipeline trained with 3 features make predictions when it receives 8 features?

In [18]:
import numpy as np
import joblib

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## Step 1: Create the Dataset

We create a dataset containing 8 input features and one target variable.

Synthetic data is used here to demonstrate the behavior of the Pipeline.

In [19]:
rng = np.random.default_rng(42)

X = rng.normal(size=(300, 8))

y = (
    2 * X[:, 0]
    - 1.5 * X[:, 1]
    + 0.7 * X[:, 2]
    + rng.normal(0, 0.2, size=300)
)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (300, 8)
y shape: (300,)


## Step 2: Split the Dataset

We split the dataset into training and testing sets.

- 80% of the data is used for training.
- 20% of the data is used for testing.

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (240, 8)
X_test: (60, 8)
y_train: (240,)
y_test: (60,)


## Step 3: Select Three Features

We select columns 0, 1, and 2 from the original dataset.

The Pipeline will be trained using only these three features.

In [21]:
selected_columns = [0, 1, 2]

X_train_3 = X_train[:, selected_columns]
X_test_3 = X_test[:, selected_columns]

print("Training features:", X_train_3.shape)
print("Testing features:", X_test_3.shape)

Training features: (240, 3)
Testing features: (60, 3)


## Step 4: Build and Train the Pipeline

The Pipeline contains two steps:

1. StandardScaler: standardizes the input features.
2. RandomForestRegressor: trains the regression model.

The Pipeline is trained using only three features.

In [22]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

pipeline.fit(X_train_3, y_train)

print("Expected number of features:", pipeline.n_features_in_)

Expected number of features: 3


## Step 5: Evaluate the Pipeline

We evaluate the trained Pipeline using test data containing the same three features used during training.

RMSE measures the typical prediction error.

In [23]:
predictions_3 = pipeline.predict(X_test_3)

rmse_3 = np.sqrt(
    mean_squared_error(y_test, predictions_3)
)

print("Prediction shape:", predictions_3.shape)
print("RMSE:", round(rmse_3, 4))

Prediction shape: (60,)
RMSE: 0.7063


## Step 6: Save the Pipeline

We save the trained Pipeline using Joblib.

The model is saved to a local file, and its file size is displayed in bytes.

In [24]:
model_path = Path("exercise1_pipeline.joblib")

joblib.dump(pipeline, model_path)

print("Model saved successfully.")
print("File size:", model_path.stat().st_size, "bytes")

Model saved successfully.
File size: 2211466 bytes


## Step 7: Load the Pipeline

We load the saved Pipeline and check how many features it expects.

In [25]:
loaded_pipeline = joblib.load(model_path)

print("Pipeline loaded successfully.")
print("Expected number of features:", loaded_pipeline.n_features_in_)

Pipeline loaded successfully.
Expected number of features: 3


## Step 8: Predict Using Eight Features

The loaded Pipeline was trained using three features.

Now, we pass test data containing all eight features and observe the result.

In [26]:
try:
    predictions_8 = loaded_pipeline.predict(X_test)

    print("Prediction succeeded.")
    print("Prediction shape:", predictions_8.shape)

except ValueError as error:
    print("Prediction failed as expected:")
    print(error)

Prediction failed as expected:
X has 8 features, but StandardScaler is expecting 3 features as input.


## Final Answer

The Pipeline cannot directly accept eight features because it was trained using only three features.

Saving and loading the Pipeline does not change the number of features expected by the model.

Passing eight features causes a ValueError.

To make a valid prediction, we must select the same three features used during training, in the same order.